# BERT fake-vs-real classifier on HF-vs-HR (PolitiFact++ & GossipCop++)

Adapts `fake-vs-real-news-detection-bert-acc-100.ipynb` (`bert-base-uncased`), pointed at the
LIFE **human-written** subset. **Task = fake-vs-real among human news:** HF (human_fake) =
**0 (fake)**, HR (human_true) = **1 (real)** — the *same cut* as `run_life_lstm.py` (§7n), so this
is a direct **BERT-vs-LSTM head-to-head**.

- The source notebook is TF/Keras, but current Colab `transformers` dropped TF support (Keras 3),
  so this runs BERT in **PyTorch** (same model/technique; the stack that ran the RoBERTa track).
- Same pipeline: clean (lowercase/stopwords/punct) → BERT tokenize → fine-tune 5 epochs,
  AdamW 1e-5, stratified 90/10→90/10 split, Fake=0/Real=1.
- **The source notebook's "100%" is a source-leakage artifact** (ISOT real news is all
  Reuters-formatted). Expect **honest, lower** numbers on LIFE data.
- **Caveats:** PolitiFact++ HF/HR is only **291 articles** (~29 test) → noisy; BERT may still
  beat the LSTM's collapse there. GossipCop++ (**12,252**) is the longer run (~10–15 min on GPU,
  all articles by default). Both are **1:2 fake:real** → watch **fake(HF=0) recall** + the
  confusion matrix, not just accuracy.
- **GPU required** (Runtime → Change runtime type → GPU).

In [1]:
!pip install -q transformers  # torch is preinstalled on Colab

In [2]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime to GPU')

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

PROJECT_DIR    = '/content/drive/MyDrive/LIFE'
DATASET_ROOT   = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset'
POLITIFACT_DIR = f'{DATASET_ROOT}/PolitiFact++'
GOSSIPCOP_DIR  = f'{DATASET_ROOT}/GossipCop++'

os.chdir(PROJECT_DIR)  # so the relative script path (and its `import run_life_lstm`) resolve
print('PolitiFact++ found:', os.path.isdir(POLITIFACT_DIR))
print('GossipCop++  found:', os.path.isdir(GOSSIPCOP_DIR))

PolitiFact++ found: True
GossipCop++  found: True


## Run the classifier

Each cell prints the class balance, per-epoch train/val accuracy, then a test
`classification_report` + confusion matrix. PolitiFact++ is quick; GossipCop++ (all ~12k
articles, 5 epochs) takes ~10–15 min on a GPU.

In [ ]:
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{POLITIFACT_DIR}" --name PolitiFact++

In [ ]:
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{GOSSIPCOP_DIR}" --name GossipCop++

## Notes
- **Task = fake-vs-real among human news** (HF=0 vs HR=1) — the same cut as the LSTM (§7n), for a
  BERT-vs-LSTM comparison. It imports `FILE_LABELS` + `load_dataframe` from `run_life_lstm.py`, so
  **sync that file to Drive too** (both live in `lstm_real_vs_fake_code/`).
- Follows the source notebook (bert-base-uncased, lr 1e-5, 5 epochs, stratified 90/10→90/10)
  but in **PyTorch** (its TF path is unavailable on current Colab), with num_labels=2 +
  cross-entropy, subsample capped at dataset size, and `--max_length 256`.
- GossipCop++ uses **all** articles by default (`--sample_size 0`); pass e.g. `--sample_size 1000`
  for the source notebook's fast subsample behavior. No model checkpoints saved.
- **Enhanced-BERT flags** (`run_life_bert.py`): `--head {plain,enhanced}` (default `plain` = unchanged
  §7o behavior), `--num_custom_layers 1`, `--hidden_dim 256`, `--head_dropout 0.1`. The paper specifies
  neither `hidden_dim` nor the dropout ("Dynamic", Table 2); `num_custom_layers=1` follows its Fig. 6.
- **Deliberate deviations from the paper** (recorded so the A/B stays honest): loss is **cross-entropy**
  — its §IV-D text and Eq. 2 say cross-entropy while Table 2 says MSE, an internal contradiction; and
  **lr 1e-5 / batch 16** are kept: Table 3's lr 0.01 would destroy BERT's pretrained weights under full
  fine-tuning, and Table 2's batch 256 would shift the optimization dynamics away from the §7o run this is
  an A/B against (it fits fine on the A100 — the reason to hold batch 16 is parity, not VRAM).
- **Not implemented** (deferred by choice): the paper's data augmentation (synonym replacement / word
  reordering to balance classes). That — not the extra layers — is the paper's actual lever for our known
  weak spot, GossipCop fake recall 0.51 caused by the 1:2 fake:real imbalance.
- Seeds are written out literally rather than looped: IPython expands `{NAME}` inside a `!` line from the
  notebook's globals, so `{POLITIFACT_DIR}` (cell 4) resolves but a loop variable does not.
- **`--select_best_val`** (off by default): tests the highest-val-accuracy epoch instead of the last one.
  The first enhanced GossipCop run showed why it is needed — val went 0.8166 / 0.8166 / **0.8347** / 0.8166 /
  **0.7405** while train accuracy climbed to 0.9343, so the reported test number (0.7313) came from the most
  overfit checkpoint, not the best one. The paper applies early stopping for the same reason (§V-C). The
  GossipCop cells below pass it; **the PolitiFact cells do not** — re-run them with the flag too if you want
  one protocol across both datasets (those runs are quick).

## Enhanced BERT (paper's custom head) — A/B vs the plain head

Runs the **Enhanced BERT** head of Oad et al., *IEEE Access* 12, 2024
(DOI 10.1109/ACCESS.2024.3491376, §IV-D) against the plain head on the **same** HF-vs-HR cut,
preprocessing, split, optimizer, lr and epochs — so the only difference is the architecture:

```
pooled [CLS] (768) -> [Linear 768x768 + ReLU] x num_custom_layers -> Dropout
                   -> Linear(768 -> hidden_dim) + ReLU -> Linear(hidden_dim -> 2)
```

**What the paper claims:** 98.23% on PolitiTweet, credited to these added layers.
**What to expect here:** the head adds only **~0.79M** parameters on top of BERT's 110M, all
downstream of the same pooled `[CLS]` vector, so **±1-2 pts** is the realistic range. The paper's
98% is also weak evidence for the architecture — on that same pak-tweets data a Keras SimpleRNN with a
*2-dimensional* embedding reaches ~0.97 and a RandomForest on raw padded token IDs 0.9768
(`97-plus-accuracy.ipynb`), i.e. PolitiTweet is trivially separable. HF/HR has no such artifact.

**Read fake(HF=0) recall and macro-F1, not accuracy**, against the plain-head baseline (§7o, seed 42):

| | Acc | macro-F1 | fake(HF=0) P/R | always-real baseline |
|---|---|---|---|---|
| PolitiFact++ (28 test) | 0.893 test / **0.76 val** | - | 0.89/0.80 | 0.643 |
| GossipCop++ (1109 test) | **0.814** | **0.76** | 0.86/**0.51** | 0.676 |

Both datasets run **3 seeds** (7/42/123); `--seed` varies the split *and* the init together, so the spread
covers both sources of variation. PolitiFact needs it most (28-article test set); GossipCop's 1109-article
test set is already stable to ~1.3pp, but on the A100 the extra runs are cheap and give mean±std on both.
Both plain arms are re-run in-session, so each pair is free of library/session drift. The GossipCop cells
also pass `--select_best_val` — the PolitiFact cells do not, so that is currently the one protocol
difference between the two datasets.

*Cells 6-7 above are the original single-run §7o repro and can be skipped.*

In [ ]:
# PolitiFact++ — PLAIN head, 3 seeds (§7o only has seed 42)
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{POLITIFACT_DIR}" --name PolitiFact++ --head plain --seed 7
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{POLITIFACT_DIR}" --name PolitiFact++ --head plain --seed 42
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{POLITIFACT_DIR}" --name PolitiFact++ --head plain --seed 123

In [ ]:
# PolitiFact++ — ENHANCED head, 3 seeds
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{POLITIFACT_DIR}" --name PolitiFact++ --head enhanced --seed 7
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{POLITIFACT_DIR}" --name PolitiFact++ --head enhanced --seed 42
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{POLITIFACT_DIR}" --name PolitiFact++ --head enhanced --seed 123

In [5]:
# GossipCop++ — PLAIN head, 3 seeds (same-session control for the §7o number)
# --select_best_val: test the best-val epoch, not the overfit last one (see Notes)
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{GOSSIPCOP_DIR}" --name GossipCop++ --head plain --seed 7 --select_best_val
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{GOSSIPCOP_DIR}" --name GossipCop++ --head plain --seed 42 --select_best_val
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{GOSSIPCOP_DIR}" --name GossipCop++ --head plain --seed 123 --select_best_val

2026-07-28 14:42:20.467442: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-28 14:42:20.540966: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
config.json: 100% 570/570 [00:00<00:00, 2.76MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 249kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 781kB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 979kB/s]
[GossipCop++] 11081 articles | fake HF(0)=3590 real HR(1)=7491 | device=cuda

model.safetensors: downloading bytes:  50% 222M/440M [00:02<00:00, 240MB/s, 17.3MB

In [6]:
# GossipCop++ — ENHANCED head, 3 seeds
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{GOSSIPCOP_DIR}" --name GossipCop++ --head enhanced --seed 7 --select_best_val
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{GOSSIPCOP_DIR}" --name GossipCop++ --head enhanced --seed 42 --select_best_val
!python lstm_real_vs_fake_code/run_life_bert.py --data_dir "{GOSSIPCOP_DIR}" --name GossipCop++ --head enhanced --seed 123 --select_best_val

2026-07-28 15:06:04.709059: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-28 15:06:04.779017: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[GossipCop++] 11081 articles | fake HF(0)=3590 real HR(1)=7491 | device=cuda
Loading weights: 100% 199/199 [00:00<00:00, 5954.72it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED